# 04 — dQ/dV curves and figures
This notebook asks: *how does differential capacity vary with voltage for the selected synthetic cycle?* It independently imports and elaborates the example, then calculates raw and Savitzky–Golay-smoothed dQ/dV in ampere-hours per volt. Positive current denotes charge, and every voltage axis is in volts. The small synthetic curve explains API and plot semantics, not a material fingerprint.


In [ ]:
import tempfile
from pathlib import Path


def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples" / "data" / "synthetic_cycling.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the gregorithm repository folder.")


PROJECT_ROOT = find_project_root()
LOCAL_FILE = None  # Optional: Path(r"D:/your-local/canonical-file.csv")
SOURCE = (
    Path(LOCAL_FILE)
    if LOCAL_FILE is not None
    else PROJECT_ROOT / "examples" / "data" / "synthetic_cycling.csv"
)
_session = tempfile.TemporaryDirectory(prefix="gregorithm-notebook-")
WORK = Path(_session.name)
print(f"Input: {SOURCE.name}")
print(f"Temporary result area: {WORK}")

In [ ]:
from gregorithm import ElaborationRequest, ImportRequest, elaborate, import_source

imported = import_source(
    ImportRequest(SOURCE, WORK / "import", "synthetic-cell", "import-job", "import-v1")
)
elaborated = elaborate(
    ElaborationRequest(
        imported.ref.path / "points" / "canonical.parquet",
        imported.report.content_hash,
        WORK / "elaboration",
        "synthetic-cell",
        "elaboration-job",
        "elaboration-v1",
    )
)
print(f"Retained rows: {imported.report.retained_rows}")
print(f"Cycles: {elaborated.cycles}; half-cycles: {elaborated.half_cycles}")

In [ ]:
from gregorithm import DqdvParameters, DqdvRequest, analyze_dqdv

parameters = DqdvParameters(
    method="interpolated_gradient",
    interpolation_value=10.0,
    edge_trim_mv=0.0,  # the synthetic half-cycles have only a few points: keep them all
    smoothing_enabled=True,
    sg_window_points=3,
    sg_poly_order=1,
)
dqdv_result = analyze_dqdv(
    DqdvRequest(
        elaborated.ref.path,
        elaborated.fingerprint,
        WORK / "dqdv",
        "synthetic-cell",
        "dqdv-job",
        "dqdv-v1",
        cycles=(1,),
        parameters=parameters,
    )
)
print(f"Curves: {dqdv_result.curves}; points: {dqdv_result.points}")

In [ ]:
import pyarrow.parquet as pq

dqdv = pq.read_table(dqdv_result.ref.path / "tables" / "dqdv.parquet")
dqdv.to_pylist()[:8]

In [ ]:
import matplotlib.pyplot as plt
import pyarrow.compute as pc

fig, ax = plt.subplots(figsize=(8, 4))
for phase in sorted(set(dqdv["phase"].to_pylist())):
    phase_table = dqdv.filter(pc.equal(dqdv["phase"], phase))
    voltage = phase_table["voltage_mid_v"].to_numpy()
    raw = phase_table["dqdv_raw_ah_per_v"].to_numpy()
    smooth = phase_table["dqdv_smoothed_ah_per_v"].to_numpy()
    ax.plot(voltage, raw, marker=".", linestyle="--", alpha=0.65, label=f"{phase} raw")
    ax.plot(voltage, smooth, linewidth=2, label=f"{phase} smoothed")
ax.set_xlabel("Voltage (V)")
ax.set_ylabel("dQ/dV (Ah/V)")
ax.set_title("Synthetic differential-capacity curves")
ax.legend()
fig

The interpolation grid has 10 % as many points as the measured half-cycle, evenly spaced in volts. Raw values are calculated output; smoothing is an explicit derived curve and must be reported with window and polynomial order. Reversals, plateaus, insufficient points, or unavailable phases should be interpreted through diagnostics rather than hidden by plotting.
